
## Tóm tắt tất cả các thay đổi

### Tổng quan
Sổ tay được sửa đổi có **5 mô hình** với một tập hợp meta-learner xếp chồng.

---

### Mô hình mới được thêm vào
| # | Mô hình | Mô tả |
|---|---|---|
| **5** | **LightGBM** (MỚI) | Xây dựng cây theo từng lá - cấu trúc khác với XGBoost (theo từng cấp độ) và CatBoost (đối xứng). Trung bình 3 hạt giống. |

### Cải tiến ở cấp độ mô hình

#### Hồi quy Logistic (Ô 3)
- `smooth=0` → **`smooth=10`** trong TargetEncoder (giảm hiện tượng quá khớp trên các cặp hiếm)
- `C=0.5` → **`C=1.0`** trong cuML LogisticRegression
- **Đã thêm mã hóa mục tiêu đơn tính năng** cùng với TE cặp (19 tính năng hiệu ứng biên bổ sung)

#### XGBoost (Ô 5)
- **Trung bình 3 hạt giống** (hạt giống: 42, 2024, 7777) — giảm phương sai
- **Tăng cường dữ liệu IBM Telco gốc** — Các cuộc thi PS được hưởng lợi từ dữ liệu nguồn gốc
- **Hơn 15 tính năng được thiết kế mới**: `tenure_sq`, `monthly_sq`, `log_tenure`, `log_total`, `log_monthly`, `n_protection`, `n_streaming`, `monthly_per_service`, `has_internet`, `has_fiber`, `contract_num`, `contract_x_tenure`, `contract_x_monthly`, `payment_freq`
- Phân nhóm `tenure_bin` chi tiết hơn (6 nhóm thay vì 5)
- Phương pháp gán giá trị cho `TotalCharges`: `fillna(0)` → **`tenure × MonthlyCharges`**
- `early_stopping_rounds` 300 → **150**

#### MLP (Ô 7)
- `EPOCHS` 10 → **100**
- `LR` 2.5e-5 → **2e-3** (tăng 100 lần — MLP dạng bảng cần LR cao hơn)
- `BATCH_SIZE` 256 → **512**
- `HIDDEN` (512, 256) → **(512, 512, 256)** (sâu hơn)
- `WARMUP_EPOCHS` 1 → **4**
- `MLP_DROPOUT` 0.30 → **0.22**
- Kích hoạt **ReLU → SiLU** (luôn tốt hơn trên dữ liệu dạng bảng)
- Làm mịn nhãn `eps` 0.02 → **0.01**

#### CatBoost (Ô 9)

- **Trung bình 3 hạt giống** + kỹ thuật đặc trưng nâng cao tương tự + tăng cường dữ liệu gốc

#### LightGBM (Ô 11) — MỚI
- Tăng trưởng theo từng lá, `num_leaves=31`, `learning_rate=0.03`
- Kỹ thuật đặc trưng nâng cao tương tự + dữ liệu gốc + 3 hạt giống Tính trung bình

### Cải tiến Tổ hợp (Ô 13)
**Trước:** Trung bình đơn giản của 3 mô hình → CV 0.9178

**Sau:** 7 chiến lược được so sánh tự động, chiến lược tốt nhất được chọn tự động:
1. Trung bình đơn giản (5 mô hình)
2. Trung bình xếp hạng
3. Trọng số tối ưu (xác suất thô, Nelder-Mead)
4. Trọng số tối ưu (chuyển đổi xếp hạng)
5. **Ghép nối LR với các đặc trưng tương tác theo cặp** (CV 10 lần)
6. **Ghép nối Ridge trên các đặc trưng xếp hạng** (CV 10 lần)

7. **Ghép nối LR đơn giản** (CV 10 lần)

### Nộp bài (Ô 15)
- Tự động áp dụng chiến lược tổ hợp nào đạt điểm cao nhất trên OOF
- Hỗ trợ tất cả 7 chiến lược bao gồm ghép nối suy luận meta-learner trên tập dữ liệu thử nghiệm

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Hồi quy Logistic - Nvidia cuML - CV 0.9160 -> 0.9162
Chúng ta tạo một cột mới từ mỗi cặp đặc trưng. Sau đó, chúng ta áp dụng mã hóa mục tiêu Nvidia cuML. Tiếp theo, chúng ta tạo các cột từ x, x^2, x^3. Cuối cùng, chúng ta thực hiện hồi quy logistic Nvidia cuML. Đây là mô hình yêu thích của tôi trong cuộc thi Kaggle playground tháng trước được mô tả [tại đây][1]. Và nó cũng hoạt động tốt trong cuộc thi playground tháng này!

[1]: https://www.kaggle.com/competitions/playground-series-s6e2/discussion/679367

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

VER = 200

import numpy as np
import pandas as pd

import cudf
import cupy as cp
from cuml.preprocessing import TargetEncoder
from cuml.linear_model import LogisticRegression as cuLogReg

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -----------------------------
# LOAD
# -----------------------------
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

# -----------------------------
# cấu hình
# -----------------------------
TARGET = "Churn"
DROP_COLS = ["customerID", "id"]
FEATURES = [c for c in train.columns if c not in DROP_COLS + [TARGET]]

N_SPLITS = 5
RANDOM_STATE = 42


TE_N_FOLDS = 5

# -----------------------------
# TARGET
# -----------------------------
y = (train[TARGET].astype(str).str.strip().str.lower() == "yes").astype(np.int32).values
y_all = y.astype(np.int32)
N = len(train)
n_feat = len(FEATURES)

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# ============================================================
# LABEL ENCODE (Tập huấn luyện và kiểm thử cùng nhau theo từng cột)
# ============================================================
def label_encode_train_test(train_df: pd.DataFrame, test_df: pd.DataFrame, cols):
    train_out = train_df.copy()
    test_out  = test_df.copy()

    for c in cols:
        tr = train_out[c]
        te = test_out[c]

        # coi dữ liệu thiếu là một loại
        tr_s = tr.astype("string").fillna("__MISSING__")
        te_s = te.astype("string").fillna("__MISSING__")

        # Điều chỉnh ánh xạ trên các giá trị kết hợp (mã ổn định trên tập huấn luyện/kiểm tra)
        all_vals = pd.concat([tr_s, te_s], axis=0)
        uniq = pd.Index(all_vals.unique())

        # map to int32 codes
        mapping = pd.Series(np.arange(len(uniq), dtype=np.int32), index=uniq)
        train_out[c] = tr_s.map(mapping).astype(np.int32)
        test_out[c]  = te_s.map(mapping).astype(np.int32)

    return train_out, test_out

train_enc, test_enc = label_encode_train_test(train[FEATURES], test[FEATURES], FEATURES)

# -----------------------------
# Chuyển tất cả các tính năng sang GPU một lần
# -----------------------------
X_all_g  = cudf.DataFrame({f: cudf.Series(train_enc[f].values).astype("int32") for f in FEATURES})
X_test_g = cudf.DataFrame({f: cudf.Series(test_enc[f].values).astype("int32") for f in FEATURES})

# -----------------------------
# Phân bổ sổ sách kế toán theo cặp đặc trưng mới
# -----------------------------
n_pair = n_feat * (n_feat - 1) // 2  # C(n,2)
pair_cols = []
pair_names = []
for i in range(n_feat):
    for j in range(i + 1, n_feat):
        f1, f2 = FEATURES[i], FEATURES[j]
        pair_cols.append((f1, f2))
        pair_names.append(f"{f1}__x__{f2}")

print(f"Num raw features: {n_feat}")
print(f"Num pairs:        {n_pair}")
print(f"Outer folds:      {N_SPLITS}")
print(f"TE inner folds:   {TE_N_FOLDS} (inside each outer fold)")

# ============================================================
# MÔ HÌNH SIÊU CẤP: LOGIT 3 (z + z^2 + z^3) tiện ích
# ============================================================
def _clip01(x, eps=1e-5):
    return np.clip(x, eps, 1.0 - eps)

def _logit(x, eps=1e-5):
    x = _clip01(x, eps)
    return np.log(x / (1.0 - x)).astype(np.float32)

def make_logit3_features(tr_m, va_m, te_m, eps=1e-5):
    z_tr = _logit(tr_m, eps=eps)
    z_va = _logit(va_m, eps=eps)
    z_te = _logit(te_m, eps=eps)
    X_tr = np.hstack([z_tr, z_tr**2, z_tr**3]).astype(np.float32)
    X_va = np.hstack([z_va, z_va**2, z_va**3]).astype(np.float32)
    X_te = np.hstack([z_te, z_te**2, z_te**3]).astype(np.float32)
    return X_tr, X_va, X_te

# -----------------------------
# Kết quả đầu ra cho mô hình meta OOOF + kiểm tra dự đoán
# -----------------------------
oof_meta  = np.zeros(N, dtype=np.float32)
pred_meta = np.zeros(len(test), dtype=np.float32)
fold_aucs = []

print("\n================ LEAK-FREE OUTER CV (PAIR TE INSIDE FOLDS) ================\n")

# ============================================================
# VÒNG LẶP CV BÊN NGOÀI:
# ============================================================
for fold, (tr_idx, va_idx) in enumerate(kf.split(np.zeros(N), y_all), 1):
    print("\n" + "=" * 90)
    print(f"[OUTER FOLD {fold:02d}/{N_SPLITS}] tr={len(tr_idx)} va={len(va_idx)}")
    print("=" * 90)

    tr_idx_g = cudf.Series(tr_idx)
    va_idx_g = cudf.Series(va_idx)

    X_tr_g = X_all_g.take(tr_idx_g)
    X_va_g = X_all_g.take(va_idx_g)

    y_tr = y_all[tr_idx].astype(np.int32)
    y_va = y_all[va_idx].astype(np.int32)
    y_tr_g = cudf.Series(y_tr)

    tr_pair = np.zeros((len(tr_idx), n_pair), dtype=np.float32)
    va_pair = np.zeros((len(va_idx), n_pair), dtype=np.float32)
    te_pair = np.zeros((len(test),  n_pair), dtype=np.float32)

    te = TargetEncoder(
        n_folds=TE_N_FOLDS,
        smooth=10,
        seed=RANDOM_STATE + fold,
        split_method="random",
        stat="mean",
        output_type="cupy",
    )

    for t, (f1, f2) in enumerate(pair_cols):
        if (t == 0) or ((t + 1) % 200 == 0) or (t + 1 == n_pair):
            print(f"  Pair {t+1:>6d}/{n_pair}: {f1} x {f2}")

        X_tr_ij_g = X_tr_g[[f1, f2]]
        X_va_ij_g = X_va_g[[f1, f2]]
        X_te_ij_g = X_test_g[[f1, f2]]

        tr_oof_cp = te.fit_transform(X_tr_ij_g, y_tr_g)
        tr_pair[:, t] = cp.asnumpy(tr_oof_cp).ravel().astype(np.float32)

        te.fit(X_tr_ij_g, y_tr_g)
        va_cp = te.transform(X_va_ij_g)
        te_cp = te.transform(X_te_ij_g)
        va_pair[:, t] = cp.asnumpy(va_cp).ravel().astype(np.float32)
        te_pair[:, t] = cp.asnumpy(te_cp).ravel().astype(np.float32)

    # ---- Mã hóa mục tiêu đơn tính năng ----
    tr_single = np.zeros((len(tr_idx), n_feat), dtype=np.float32)
    va_single = np.zeros((len(va_idx), n_feat), dtype=np.float32)
    te_single = np.zeros((len(test),  n_feat), dtype=np.float32)

    te_single_enc = TargetEncoder(
        n_folds=TE_N_FOLDS,
        smooth=10,
        seed=RANDOM_STATE + fold + 100,
        split_method="random",
        stat="mean",
        output_type="cupy",
    )

    for t, f in enumerate(FEATURES):
        X_tr_f_g = X_tr_g[[f]]
        X_va_f_g = X_va_g[[f]]
        X_te_f_g = X_test_g[[f]]

        tr_oof_s = te_single_enc.fit_transform(X_tr_f_g, y_tr_g)
        tr_single[:, t] = cp.asnumpy(tr_oof_s).ravel().astype(np.float32)

        te_single_enc.fit(X_tr_f_g, y_tr_g)
        va_s = te_single_enc.transform(X_va_f_g)
        te_s = te_single_enc.transform(X_te_f_g)
        va_single[:, t] = cp.asnumpy(va_s).ravel().astype(np.float32)
        te_single[:, t] = cp.asnumpy(te_s).ravel().astype(np.float32)

    print(f"  Single-feature TEs: {n_feat}")

    # ---- Kết hợp cặp TE + TE đơn ----
    tr_all_te = np.hstack([tr_pair, tr_single])
    va_all_te = np.hstack([va_pair, va_single])
    te_all_te = np.hstack([te_pair, te_single])

    X_tr_raw, X_va_raw, X_te_raw = make_logit3_features(tr_all_te, va_all_te, te_all_te, eps=1e-5)
    print("  Raw dim (pairs+singles):", tr_all_te.shape[1])
    print("  Logit3 dim:", X_tr_raw.shape[1])

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw).astype(np.float32)
    X_va = scaler.transform(X_va_raw).astype(np.float32)
    X_te = scaler.transform(X_te_raw).astype(np.float32)

    meta = cuLogReg(
        penalty="l2",
        C=1.0,
        max_iter=4000,
        tol=1e-4,
        fit_intercept=True,
        verbose=0,
    )

    meta.fit(cp.asarray(X_tr), cp.asarray(y_tr))

    oof_va = cp.asnumpy(meta.predict_proba(cp.asarray(X_va))[:, 1]).astype(np.float32)
    oof_meta[va_idx] = oof_va

    fold_auc = roc_auc_score(y_va, oof_va)
    fold_aucs.append(float(fold_auc))
    print(f"  [logit3] Fold {fold:02d} AUC: {fold_auc:.6f}")

    pred_meta += cp.asnumpy(meta.predict_proba(cp.asarray(X_te))[:, 1]).astype(np.float32) / N_SPLITS

# -----------------------------
# CÁC CHỈ SỐ CUỐI CÙNG
# -----------------------------
meta_auc = roc_auc_score(y_all, oof_meta)
print("\n================ FINAL ================\n")
print("[logit3] OOF AUC:", f"{meta_auc:.6f}")
print("[logit3] Fold AUCs:", [float(f"{a:.6f}") for a in fold_aucs])

# -----------------------------
# SAVE
# -----------------------------
np.save(f"oof_tepair_logit3_v{VER}.npy", oof_meta.astype(np.float32))
np.save(f"pred_tepair_logit3_v{VER}.npy", pred_meta.astype(np.float32))

print(f"\nSaved:")
print(f"  oof_tepair_logit3_v{VER}.npy")
print(f"  pred_tepair_logit3_v{VER}.npy")

# XGBoost - GPU - CV 0.9166 -> 0.91614
We train a basic XGBoost model with the data as is. For categorical features, we use XGB's built in `enable_categorical=True`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

VER = 103

import numpy as np
import pandas as pd
import gc

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# -----------------------------
# CONFIG
# -----------------------------
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
ORIG_PATH  = "/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv"

TARGET = "Churn"
DROP_COLS = ["customerID", "id"]

N_SPLITS = 5
SEEDS = [42, 2024, 7777]  # 3-seed averaging

# -----------------------------
# LOAD + ORIGINAL DATA AUGMENTATION
# -----------------------------
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

assert TARGET in train_df.columns
assert TARGET not in test_df.columns

# Try augmenting with original IBM Telco data
try:
    orig_df = pd.read_csv(ORIG_PATH)
    if TARGET in orig_df.columns:
        common_cols = [c for c in train_df.columns if c in orig_df.columns]
        orig_df = orig_df[common_cols].copy()
        # Fill missing columns with NaN
        for c in train_df.columns:
            if c not in orig_df.columns:
                orig_df[c] = np.nan
        orig_df = orig_df[train_df.columns]
        train_df = pd.concat([train_df, orig_df], axis=0, ignore_index=True)
        print(f"Original data loaded! Combined train shape: {train_df.shape}")
    else:
        print("Original data found but missing target, skipping.")
except Exception as e:
    print(f"No original data found ({e}), using competition data only.")

# Target
y = (train_df[TARGET].astype(str).str.strip().str.lower() == "yes").astype(np.int32).values

X_train_raw = train_df.drop(columns=[TARGET]).copy()
X_test_raw  = test_df.copy()

# -----------------------------
# ENHANCED PREPROCESSING
# -----------------------------
def preprocess_together(X_tr: pd.DataFrame, X_te: pd.DataFrame):
    X_tr = X_tr.copy()
    X_te = X_te.copy()

    X_tr = X_tr.drop(columns=[c for c in DROP_COLS if c in X_tr.columns], errors="ignore")
    X_te = X_te.drop(columns=[c for c in DROP_COLS if c in X_te.columns], errors="ignore")

    n_tr = len(X_tr)
    X_all = pd.concat([X_tr, X_te], axis=0, ignore_index=True)

    # ---- numeric fixes ----
    if "TotalCharges" in X_all.columns:
        X_all["TotalCharges"] = pd.to_numeric(X_all["TotalCharges"], errors="coerce")
        mask = X_all["TotalCharges"].isna()
        X_all.loc[mask, "TotalCharges"] = (
            X_all.loc[mask, "tenure"].fillna(0) * X_all.loc[mask, "MonthlyCharges"].fillna(0)
        )
        X_all["TotalCharges"] = X_all["TotalCharges"].fillna(0.0).astype("float32")
    for c in ["tenure", "MonthlyCharges"]:
        if c in X_all.columns:
            X_all[c] = pd.to_numeric(X_all[c], errors="coerce")

    # ---- Feature Engineering (ENHANCED) ----
    X_all["tenure_x_monthly"] = X_all["tenure"] * X_all["MonthlyCharges"]
    X_all["avg_monthly"] = X_all["TotalCharges"] / (X_all["tenure"] + 1)
    X_all["charge_diff"] = X_all["MonthlyCharges"] - X_all["avg_monthly"]
    X_all["charge_ratio"] = X_all["MonthlyCharges"] / (X_all["TotalCharges"] + 1)

    # Polynomial / log features
    X_all["tenure_sq"] = X_all["tenure"] ** 2
    X_all["monthly_sq"] = X_all["MonthlyCharges"] ** 2
    X_all["log_tenure"] = np.log1p(X_all["tenure"])
    X_all["log_total"] = np.log1p(X_all["TotalCharges"])
    X_all["log_monthly"] = np.log1p(X_all["MonthlyCharges"])

    # Service counts
    service_cols = ["PhoneService","MultipleLines","OnlineSecurity","OnlineBackup",
                    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
    svc_present = [c for c in service_cols if c in X_all.columns]
    for c in svc_present:
        X_all[f"{c}_flag"] = (X_all[c].astype(str).str.lower() == "yes").astype(np.int8)
    X_all["n_services"] = X_all[[f"{c}_flag" for c in svc_present]].sum(axis=1)

    # Sub-counts: protection vs streaming
    prot_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
    pc_present = [c for c in prot_cols if c in X_all.columns]
    X_all["n_protection"] = X_all[[f"{c}_flag" for c in pc_present]].sum(axis=1)

    stream_cols = ["StreamingTV", "StreamingMovies"]
    sc_present = [c for c in stream_cols if c in X_all.columns]
    X_all["n_streaming"] = X_all[[f"{c}_flag" for c in sc_present]].sum(axis=1)

    X_all = X_all.drop(columns=[f"{c}_flag" for c in svc_present])

    # Monthly per service
    X_all["monthly_per_service"] = X_all["MonthlyCharges"] / (X_all["n_services"] + 1)

    # Internet type flags
    if "InternetService" in X_all.columns:
        X_all["has_internet"] = (X_all["InternetService"].astype(str).str.lower() != "no").astype(np.int8)
        X_all["has_fiber"] = (X_all["InternetService"].astype(str).str.lower() == "fiber optic").astype(np.int8)

    # Contract encoding + interaction
    if "Contract" in X_all.columns:
        contract_map = {"month-to-month": 0, "one year": 1, "two year": 2}
        X_all["contract_num"] = X_all["Contract"].astype(str).str.lower().map(contract_map).fillna(0).astype(np.int8)
        X_all["contract_x_tenure"] = X_all["contract_num"] * X_all["tenure"]
        X_all["contract_x_monthly"] = X_all["contract_num"] * X_all["MonthlyCharges"]

    # Payment method frequency
    if "PaymentMethod" in X_all.columns:
        freq = X_all["PaymentMethod"].value_counts(normalize=True)
        X_all["payment_freq"] = X_all["PaymentMethod"].map(freq).astype(np.float32)

    # Tenure bins
    X_all["tenure_bin"] = pd.cut(X_all["tenure"], bins=[-1,6,12,24,48,72,999], labels=False).astype("float32")

    # ---- categorical dtypes ----
    cat_cols = set(X_all.select_dtypes(include=["object"]).columns.tolist())
    cat_cols |= set(X_all.select_dtypes(include=["category"]).columns.tolist())

    for c in sorted(cat_cols):
        s = X_all[c]
        if str(s.dtype) == "category":
            s = s.astype("object")
        non_null = s[~pd.isna(s)]
        cats = pd.Index(non_null.map(lambda v: str(v)).unique(), dtype="object").sort_values()
        dtype = pd.CategoricalDtype(categories=cats, ordered=False)
        X_all[c] = s.map(lambda v: np.nan if pd.isna(v) else str(v)).astype(dtype)

    X_tr2 = X_all.iloc[:n_tr].reset_index(drop=True)
    X_te2 = X_all.iloc[n_tr:].reset_index(drop=True)
    return X_tr2, X_te2

X_train, X_test = preprocess_together(X_train_raw, X_test_raw)
print(f"XGBoost features: {X_train.shape[1]}")

# NaN info
if X_train.isna().any().any() or X_test.isna().any().any():
    n_nan_tr = int(X_train.isna().sum().sum())
    n_nan_te = int(X_test.isna().sum().sum())
    print(f"Info: NaNs -> train: {n_nan_tr}, test: {n_nan_te}. XGBoost handles NaNs natively.")

# -----------------------------
# MODEL PARAMS
# -----------------------------
xgb_params = dict(
    n_estimators=100_000,
    learning_rate=0.0584204163313777,
    max_depth=4,
    min_child_weight=7.978130445075834,
    subsample=0.8689106822805999,
    colsample_bytree=0.6126759342594297,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    n_jobs=-1,
    early_stopping_rounds=150,
    device="cuda",
)

# -----------------------------
# 3-SEED AVERAGED CV
# -----------------------------
oof_accum = np.zeros(len(X_train), dtype=np.float64)
test_accum = np.zeros(len(X_test), dtype=np.float64)

for seed_idx, SEED in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"SEED {seed_idx+1}/{len(SEEDS)}: {SEED}")
    print(f"{'='*60}")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_seed = np.zeros(len(X_train), dtype=np.float32)
    test_seed = np.zeros(len(X_test), dtype=np.float32)
    fold_aucs = []

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y), 1):
        X_trn, X_val = X_train.iloc[trn_idx], X_train.iloc[val_idx]
        y_trn, y_val = y[trn_idx], y[val_idx]

        model = XGBClassifier(**xgb_params, random_state=SEED)
        model.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=500)

        p_val = model.predict_proba(X_val)[:, 1].astype(np.float32)
        oof_seed[val_idx] = p_val

        auc = roc_auc_score(y_val, p_val)
        fold_aucs.append(auc)
        print(f"  Fold {fold}/{N_SPLITS} AUC = {auc:.6f}")

        test_seed += model.predict_proba(X_test)[:, 1].astype(np.float32) / N_SPLITS
        del model; gc.collect()

    seed_auc = roc_auc_score(y, oof_seed)
    print(f"  Seed {SEED} OOF AUC: {seed_auc:.6f}")

    oof_accum += oof_seed
    test_accum += test_seed

oof = (oof_accum / len(SEEDS)).astype(np.float32)
test_pred = (test_accum / len(SEEDS)).astype(np.float32)

final_auc = roc_auc_score(y, oof)
print(f"\n{'='*60}")
print(f"3-Seed Averaged XGBoost OOF AUC: {final_auc:.6f}")
print(f"{'='*60}")

# SAVE
np.save(f"oof_pred_proba_v{VER}.npy", oof)
np.save(f"test_pred_proba_v{VER}.npy", test_pred)
print(f"\nSaved: oof_pred_proba_v{VER}.npy, test_pred_proba_v{VER}.npy")

# PyTorch MLP - GPU - CV 0.9164 -> 0.91644
We train a simple PyTorch MLP. We create embeddings for the categorical columns. For the numeric columns we create both numeric column and categorical with embeddings. When converting numeric column to categorical, we replace rare values with their closest non-rare neightbor. This reduces cardinality and helps the model find meaningful patterns. 

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

VER = 312

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")  # PyTorch 2.x

# -----------------------------
# Repro (optional)
# -----------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
print("train:", train.shape, "test:", test.shape)

# -----------------------------
# Config
# -----------------------------
TARGET_COL = "Churn"
FEATS = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines',
         'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
         'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
         'MonthlyCharges', 'TotalCharges']

NUMS = ["tenure", "MonthlyCharges", "TotalCharges"]
CAT_FEATS = [c for c in FEATS if c not in NUMS]

# Rare handling for numeric->categorical proxy
RARE_MIN_COUNT = 25

N_SPLITS = 5
RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 100
BATCH_SIZE = 512

LR = 2e-3
PATIENCE = 10

# stronger reg (helps slow down / avoid sharp minima)
WEIGHT_DECAY = 3e-4

EMB_DROPOUT = 0.10
MLP_DROPOUT = 0.22
HIDDEN = (512, 512, 256)

# LR schedule
WARMUP_EPOCHS = 4
ETA_MIN = LR * 0.05

# -----------------------------
# Helpers
# -----------------------------
def make_vocab_maps(train_df: pd.DataFrame, cols):
    """Build value->id maps per column using ONLY training fold data. Reserve 0 for UNK."""
    maps = {}
    sizes = {}
    for c in cols:
        uniq = pd.Series(train_df[c].values).astype(str).unique().tolist()
        v2i = {v: i + 1 for i, v in enumerate(uniq)}  # 1..K
        maps[c] = v2i
        sizes[c] = len(v2i) + 1  # + UNK(0)
    return maps, sizes

def encode_with_maps(df: pd.DataFrame, cols, maps):
    """Encode df[cols] into int64 array [N, C] using per-column maps. Unknown -> 0."""
    X = np.zeros((len(df), len(cols)), dtype=np.int64)
    for j, c in enumerate(cols):
        v2i = maps[c]
        s = pd.Series(df[c].values).astype(str).map(v2i).fillna(0).astype(np.int64).values
        X[:, j] = s
    return X

def emb_dim_from_card(card: int) -> int:
    """
    Avoid overfitting: don't use sqrt(card) for huge-card cols.
    This is a safer "tabular embedding" rule that still allows >16 dims.
    """
    d = int(round(1.8 * (card ** 0.25)))
    return int(np.clip(d, 4, 64))

def build_numeric_snapper(train_series: pd.Series, rare_min_count: int):
    """
    From train-fold only:
      - compute value counts
      - define frequent values (count >= rare_min_count)
      - mapping for any series: snap rare -> nearest frequent
      - rare flag: 1 if rare (or nan), 0 if frequent
    """
    s = pd.to_numeric(train_series, errors="coerce").astype(np.float32)
    vc = pd.Series(s).value_counts(dropna=False)

    frequent_vals = vc[vc >= rare_min_count].index.values
    frequent_vals = np.array([v for v in frequent_vals if pd.notna(v)], dtype=np.float32)

    if frequent_vals.size == 0:
        frequent_vals = np.array(pd.Series(s.dropna()).unique(), dtype=np.float32)

    frequent_vals = np.sort(frequent_vals)
    frequent_set = set(frequent_vals.tolist())

    def transform(series_any: pd.Series):
        x = pd.to_numeric(series_any, errors="coerce").astype(np.float32).values
        is_nan = np.isnan(x)

        # rare flag (nan => rare)
        is_rare = np.ones_like(x, dtype=np.int32)
        # vectorized-ish: membership check via python set (ok for 1D)
        for i, v in enumerate(x):
            if np.isnan(v):
                is_rare[i] = 1
            else:
                is_rare[i] = 0 if (float(v) in frequent_set) else 1

        x_snapped = x.copy()
        if frequent_vals.size > 0:
            idx_snap = np.where((~is_nan) & (is_rare == 1))[0]
            if idx_snap.size > 0:
                v = x[idx_snap]
                pos = np.searchsorted(frequent_vals, v)
                pos = np.clip(pos, 0, len(frequent_vals) - 1)
                left = np.clip(pos - 1, 0, len(frequent_vals) - 1)
                right = pos
                left_vals = frequent_vals[left]
                right_vals = frequent_vals[right]
                choose_right = (np.abs(v - right_vals) <= np.abs(v - left_vals))
                nearest = np.where(choose_right, right_vals, left_vals)
                x_snapped[idx_snap] = nearest.astype(np.float32)

        return x_snapped.astype(np.float32), is_rare.astype(np.int32)

    return transform

# -----------------------------
# Dataset
# -----------------------------
class TabMixDataset(Dataset):
    def __init__(self, X_cat, X_num, y=None):
        self.Xc = torch.as_tensor(X_cat, dtype=torch.long)
        self.Xn = torch.as_tensor(X_num, dtype=torch.float32)
        self.y  = None if y is None else torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.Xc.shape[0]

    def __getitem__(self, idx):
        if self.y is None:
            return self.Xc[idx], self.Xn[idx]
        return self.Xc[idx], self.Xn[idx], self.y[idx]

# -----------------------------
# Model
# -----------------------------
class EmbMLP_Mixed(nn.Module):
    def __init__(self, cat_cardinals, n_num, hidden=(256, 128), emb_dropout=0.1, mlp_dropout=0.2):
        super().__init__()
        self.n_cat = len(cat_cardinals)

        self.emb_layers = nn.ModuleList()
        emb_out_dim = 0
        for card in cat_cardinals:
            d = emb_dim_from_card(card)
            self.emb_layers.append(nn.Embedding(num_embeddings=card, embedding_dim=d))
            emb_out_dim += d

        self.emb_drop = nn.Dropout(emb_dropout)

        in_dim = emb_out_dim + n_num
        layers = []
        for h in hidden:
            layers += [
                nn.Linear(in_dim, h),
                nn.LayerNorm(h),     # <-- helps generalization, slows "overfit drift"
                nn.SiLU(inplace=True),
                nn.Dropout(mlp_dropout),
            ]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.mlp = nn.Sequential(*layers)

        for emb in self.emb_layers:
            nn.init.normal_(emb.weight, mean=0.0, std=0.02)

    def forward(self, x_cat, x_num):
        embs = []
        for j, emb in enumerate(self.emb_layers):
            embs.append(emb(x_cat[:, j]))
        z = torch.cat(embs, dim=1) if len(embs) else None
        if z is None:
            z = x_num
        else:
            z = self.emb_drop(z)
            z = torch.cat([z, x_num], dim=1)
        logit = self.mlp(z).squeeze(1)
        return logit

# -----------------------------
# Train / Eval
# -----------------------------
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    out = []
    for batch in loader:
        if len(batch) == 3:
            xc, xn, _ = batch
        else:
            xc, xn = batch
        xc = xc.to(DEVICE, non_blocking=True)
        xn = xn.to(DEVICE, non_blocking=True)
        logit = model(xc, xn)
        prob = torch.sigmoid(logit).detach().cpu().numpy()
        out.append(prob)
    return np.concatenate(out)

class SmoothBCE(nn.Module):
    def __init__(self, eps=0.01):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        targets = targets * (1 - self.eps) + 0.5 * self.eps
        return nn.functional.binary_cross_entropy_with_logits(logits, targets)

def train_one_fold(Xc_tr, Xn_tr, y_tr, Xc_va, Xn_va, y_va, cat_cardinals):
    model = EmbMLP_Mixed(
        cat_cardinals,
        n_num=Xn_tr.shape[1],
        hidden=HIDDEN,
        emb_dropout=EMB_DROPOUT,
        mlp_dropout=MLP_DROPOUT
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = SmoothBCE(0.01)

    dl_tr = DataLoader(
        TabMixDataset(Xc_tr, Xn_tr, y_tr),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    dl_va = DataLoader(
        TabMixDataset(Xc_va, Xn_va, y_va),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )

    # Cosine AFTER warmup
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=ETA_MIN
    )

    best_auc = -1.0
    best_state = None
    bad = 0

    print(f"  -> starting training: n_tr={len(Xc_tr):,} n_va={len(Xc_va):,} batches/tr={len(dl_tr):,}")

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()

        # -------- warmup LR (linear ramp) --------
        if epoch <= WARMUP_EPOCHS:
            # ramp from 0.1*LR -> LR
            warm_lr = LR * (0.1 + 0.9 * (epoch / WARMUP_EPOCHS))
            for pg in opt.param_groups:
                pg["lr"] = warm_lr

        running_loss = 0.0
        n_seen = 0

        for xc, xn, yb in dl_tr:
            xc = xc.to(DEVICE, non_blocking=True)
            xn = xn.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logit = model(xc, xn)
            loss = loss_fn(logit, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            bs = xc.size(0)
            running_loss += loss.item() * bs
            n_seen += bs

        # step cosine only after warmup
        if epoch > WARMUP_EPOCHS:
            sched.step()

        cur_lr = opt.param_groups[0]["lr"]
        train_loss = running_loss / max(1, n_seen)

        p_va = predict_proba(model, dl_va)
        auc = roc_auc_score(y_va, p_va)

        dt = time.time() - t0
        print(f"     epoch {epoch:03d} | lr {cur_lr:.2e} | loss {train_loss:.5f} | val_auc {auc:.6f} | {dt:.1f}s")

        if auc > best_auc + 1e-6:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"     early stopping (no improvement for {PATIENCE} epochs). best_auc={best_auc:.6f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_auc

# -----------------------------
# CV + TEST inference
# -----------------------------
y = train[TARGET_COL].values
if y.dtype == object:
    y = pd.Series(y).map({"Yes": 1, "No": 0}).values
y = y.astype(np.float32)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof = np.zeros(len(train), dtype=np.float32)
pred_test = np.zeros(len(test), dtype=np.float32)
fold_aucs = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train)), y), 1):
    fold_t0 = time.time()
    print(f"\n================ Fold {fold}/{N_SPLITS} ================")

    tr_df = train.iloc[tr_idx].reset_index(drop=True)
    va_df = train.iloc[va_idx].reset_index(drop=True)

    # -------------------------
    # Build numeric categorical proxies (fold-safe)
    # -------------------------
    tr_num_cat = {}
    va_num_cat = {}
    te_num_cat = {}
    tr_rare = {}
    va_rare = {}
    te_rare = {}

    for col in NUMS:
        snapper = build_numeric_snapper(tr_df[col], rare_min_count=RARE_MIN_COUNT)
        tr_snap, tr_israre = snapper(tr_df[col])
        va_snap, va_israre = snapper(va_df[col])
        te_snap, te_israre = snapper(test[col])

        tr_num_cat[f"{col}__cat"] = tr_snap.astype(np.float32)
        va_num_cat[f"{col}__cat"] = va_snap.astype(np.float32)
        te_num_cat[f"{col}__cat"] = te_snap.astype(np.float32)

        tr_rare[f"{col}__is_rare"] = tr_israre.astype(np.int32)
        va_rare[f"{col}__is_rare"] = va_israre.astype(np.int32)
        te_rare[f"{col}__is_rare"] = te_israre.astype(np.int32)

    # Build fold dataframes for categorical encoding
    tr_cat_df = tr_df[CAT_FEATS].copy()
    va_cat_df = va_df[CAT_FEATS].copy()
    te_cat_df = test[CAT_FEATS].copy()

    # Add numeric categorical proxies + rare flags
    for col in NUMS:
        tr_cat_df[f"{col}__cat"] = pd.Series(tr_num_cat[f"{col}__cat"]).astype(str).values
        va_cat_df[f"{col}__cat"] = pd.Series(va_num_cat[f"{col}__cat"]).astype(str).values
        te_cat_df[f"{col}__cat"] = pd.Series(te_num_cat[f"{col}__cat"]).astype(str).values

        tr_cat_df[f"{col}__is_rare"] = tr_rare[f"{col}__is_rare"]
        va_cat_df[f"{col}__is_rare"] = va_rare[f"{col}__is_rare"]
        te_cat_df[f"{col}__is_rare"] = te_rare[f"{col}__is_rare"]

    CAT_ALL = list(tr_cat_df.columns)

    # ---- categorical vocab on TRAIN fold only ----
    maps, sizes = make_vocab_maps(tr_cat_df, CAT_ALL)
    cat_cardinals = [sizes[c] for c in CAT_ALL]

    Xc_tr = encode_with_maps(tr_cat_df, CAT_ALL, maps)
    Xc_va = encode_with_maps(va_cat_df, CAT_ALL, maps)
    Xc_te = encode_with_maps(te_cat_df, CAT_ALL, maps)

    # ---- numeric float: original NUMS standardized on TRAIN fold only ----
    Xn_tr_raw = tr_df[NUMS].astype(np.float32).values
    Xn_va_raw = va_df[NUMS].astype(np.float32).values
    Xn_te_raw = test[NUMS].astype(np.float32).values

    scaler = StandardScaler()
    Xn_tr = scaler.fit_transform(Xn_tr_raw).astype(np.float32)
    Xn_va = scaler.transform(Xn_va_raw).astype(np.float32)
    Xn_te = scaler.transform(Xn_te_raw).astype(np.float32)

    y_tr = y[tr_idx]
    y_va = y[va_idx]

    model, best_auc = train_one_fold(Xc_tr, Xn_tr, y_tr, Xc_va, Xn_va, y_va, cat_cardinals)
    fold_aucs.append(best_auc)

    # OOF
    dl_va = DataLoader(TabMixDataset(Xc_va, Xn_va, y_va), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    p_va = predict_proba(model, dl_va)
    oof[va_idx] = p_va

    # TEST
    dl_te = DataLoader(TabMixDataset(Xc_te, Xn_te, None), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    p_te = predict_proba(model, dl_te).astype(np.float32)
    pred_test += p_te / N_SPLITS

    print(f"[Fold {fold}] best val AUC: {best_auc:.6f} | fold time {time.time()-fold_t0:.1f}s")

cv_auc = roc_auc_score(y, oof)
print("\n====================")
print("Fold AUCs:", [f"{a:.6f}" for a in fold_aucs])
print(f"OOF CV AUC: {cv_auc:.6f}")
print("====================")

np.save(f"oof_nn_v{VER}.npy", oof.astype(np.float32))
np.save(f"pred_nn_v{VER}.npy", pred_test.astype(np.float32))
print(f"\nSaved:\n  oof_nn_v{VER}.npy\n  pred_nn_v{VER}.npy")

# CatBoost - GPU - 4th Diverse Model - CV 0.9155
CatBoost provides native ordered target encoding for categoricals (structurally different from XGBoost) and adds real diversity to the ensemble. We reuse the same feature engineering as XGBoost plus CatBoost's built-in categorical handling.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

VER = 401

import numpy as np
import pandas as pd
import gc

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

# -----------------------------
# CONFIG
# -----------------------------
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
ORIG_PATH  = "/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv"

TARGET = "Churn"
DROP_COLS = ["customerID", "id"]

N_SPLITS = 5
SEEDS = [42, 2024, 7777]

# -----------------------------
# LOAD + ORIGINAL DATA
# -----------------------------
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

try:
    orig_df = pd.read_csv(ORIG_PATH)
    if TARGET in orig_df.columns:
        common_cols = [c for c in train_df.columns if c in orig_df.columns]
        orig_df = orig_df[common_cols].copy()
        for c in train_df.columns:
            if c not in orig_df.columns:
                orig_df[c] = np.nan
        orig_df = orig_df[train_df.columns]
        train_df = pd.concat([train_df, orig_df], axis=0, ignore_index=True)
        print(f"Original data loaded! Combined train shape: {train_df.shape}")
except Exception as e:
    print(f"No original data ({e}), using competition data only.")

y = (train_df[TARGET].astype(str).str.strip().str.lower() == "yes").astype(np.int32).values

# -----------------------------
# ENHANCED PREPROCESSING
# -----------------------------
def preprocess_catboost(X_tr: pd.DataFrame, X_te: pd.DataFrame):
    X_tr = X_tr.copy()
    X_te = X_te.copy()
    X_tr = X_tr.drop(columns=[c for c in DROP_COLS if c in X_tr.columns], errors="ignore")
    X_te = X_te.drop(columns=[c for c in DROP_COLS if c in X_te.columns], errors="ignore")

    n_tr = len(X_tr)
    X_all = pd.concat([X_tr, X_te], axis=0, ignore_index=True)

    # Numeric fixes
    if "TotalCharges" in X_all.columns:
        X_all["TotalCharges"] = pd.to_numeric(X_all["TotalCharges"], errors="coerce")
        mask = X_all["TotalCharges"].isna()
        X_all.loc[mask, "TotalCharges"] = (
            X_all.loc[mask, "tenure"].fillna(0) * X_all.loc[mask, "MonthlyCharges"].fillna(0)
        )
        X_all["TotalCharges"] = X_all["TotalCharges"].fillna(0.0).astype("float32")
    for c in ["tenure", "MonthlyCharges"]:
        if c in X_all.columns:
            X_all[c] = pd.to_numeric(X_all[c], errors="coerce")

    # ---- Enhanced Feature Engineering ----
    X_all["tenure_x_monthly"] = X_all["tenure"] * X_all["MonthlyCharges"]
    X_all["avg_monthly"] = X_all["TotalCharges"] / (X_all["tenure"] + 1)
    X_all["charge_diff"] = X_all["MonthlyCharges"] - X_all["avg_monthly"]
    X_all["charge_ratio"] = X_all["MonthlyCharges"] / (X_all["TotalCharges"] + 1)

    X_all["tenure_sq"] = X_all["tenure"] ** 2
    X_all["monthly_sq"] = X_all["MonthlyCharges"] ** 2
    X_all["log_tenure"] = np.log1p(X_all["tenure"])
    X_all["log_total"] = np.log1p(X_all["TotalCharges"])
    X_all["log_monthly"] = np.log1p(X_all["MonthlyCharges"])

    service_cols = ["PhoneService","MultipleLines","OnlineSecurity","OnlineBackup",
                    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
    svc_present = [c for c in service_cols if c in X_all.columns]
    for c in svc_present:
        X_all[f"{c}_flag"] = (X_all[c].astype(str).str.lower() == "yes").astype(np.int8)
    X_all["n_services"] = X_all[[f"{c}_flag" for c in svc_present]].sum(axis=1)

    prot_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
    pc_present = [c for c in prot_cols if c in X_all.columns]
    X_all["n_protection"] = X_all[[f"{c}_flag" for c in pc_present]].sum(axis=1)

    stream_cols = ["StreamingTV", "StreamingMovies"]
    sc_present = [c for c in stream_cols if c in X_all.columns]
    X_all["n_streaming"] = X_all[[f"{c}_flag" for c in sc_present]].sum(axis=1)
    X_all = X_all.drop(columns=[f"{c}_flag" for c in svc_present])

    X_all["monthly_per_service"] = X_all["MonthlyCharges"] / (X_all["n_services"] + 1)

    if "InternetService" in X_all.columns:
        X_all["has_internet"] = (X_all["InternetService"].astype(str).str.lower() != "no").astype(np.int8)
        X_all["has_fiber"] = (X_all["InternetService"].astype(str).str.lower() == "fiber optic").astype(np.int8)

    if "Contract" in X_all.columns:
        contract_map = {"month-to-month": 0, "one year": 1, "two year": 2}
        X_all["contract_num"] = X_all["Contract"].astype(str).str.lower().map(contract_map).fillna(0).astype(np.int8)
        X_all["contract_x_tenure"] = X_all["contract_num"] * X_all["tenure"]
        X_all["contract_x_monthly"] = X_all["contract_num"] * X_all["MonthlyCharges"]

    if "PaymentMethod" in X_all.columns:
        freq = X_all["PaymentMethod"].value_counts(normalize=True)
        X_all["payment_freq"] = X_all["PaymentMethod"].map(freq).astype(np.float32)

    X_all["tenure_bin"] = pd.cut(X_all["tenure"], bins=[-1,6,12,24,48,72,999], labels=False).astype("float32")

    # CatBoost categorical handling
    cat_cols = list(X_all.select_dtypes(include=["object", "category"]).columns)
    for c in cat_cols:
        X_all[c] = X_all[c].astype(str).fillna("__MISSING__")

    X_tr2 = X_all.iloc[:n_tr].reset_index(drop=True)
    X_te2 = X_all.iloc[n_tr:].reset_index(drop=True)
    return X_tr2, X_te2, cat_cols

X_train_raw = train_df.drop(columns=[TARGET]).copy()
X_test_raw  = test_df.copy()
X_train, X_test, cat_feature_names = preprocess_catboost(X_train_raw, X_test_raw)
cat_feature_indices = [X_train.columns.get_loc(c) for c in cat_feature_names]
print(f"CatBoost features: {X_train.shape[1]}, categorical: {len(cat_feature_names)}")

# -----------------------------
# 3-SEED AVERAGED CV
# -----------------------------
oof_accum = np.zeros(len(X_train), dtype=np.float64)
test_accum = np.zeros(len(X_test), dtype=np.float64)

for seed_idx, SEED in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"SEED {seed_idx+1}/{len(SEEDS)}: {SEED}")
    print(f"{'='*60}")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_seed = np.zeros(len(X_train), dtype=np.float32)
    test_seed = np.zeros(len(X_test), dtype=np.float32)
    fold_aucs = []

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y), 1):
        X_trn, X_val = X_train.iloc[trn_idx], X_train.iloc[val_idx]
        y_trn, y_val = y[trn_idx], y[val_idx]

        model = CatBoostClassifier(
            iterations=10000,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3.0,
            random_seed=SEED,
            task_type="GPU",
            eval_metric="AUC",
            early_stopping_rounds=200,
            verbose=500,
            cat_features=cat_feature_indices,
            bootstrap_type="Bayesian",
            bagging_temperature=0.8,
            grow_policy="SymmetricTree",
        )
        model.fit(X_trn, y_trn, eval_set=(X_val, y_val), verbose=500)

        p_val = model.predict_proba(X_val)[:, 1].astype(np.float32)
        oof_seed[val_idx] = p_val

        auc = roc_auc_score(y_val, p_val)
        fold_aucs.append(auc)
        print(f"  Fold {fold}/{N_SPLITS} AUC = {auc:.6f}")

        test_seed += model.predict_proba(X_test)[:, 1].astype(np.float32) / N_SPLITS
        del model; gc.collect()

    seed_auc = roc_auc_score(y, oof_seed)
    print(f"  Seed {SEED} OOF AUC: {seed_auc:.6f}")

    oof_accum += oof_seed
    test_accum += test_seed

oof = (oof_accum / len(SEEDS)).astype(np.float32)
test_pred = (test_accum / len(SEEDS)).astype(np.float32)

final_auc = roc_auc_score(y, oof)
print(f"\n{'='*60}")
print(f"3-Seed Averaged CatBoost OOF AUC: {final_auc:.6f}")
print(f"{'='*60}")

np.save(f"oof_catboost_v{VER}.npy", oof)
np.save(f"test_catboost_v{VER}.npy", test_pred)
print(f"\nSaved: oof_catboost_v{VER}.npy, test_catboost_v{VER}.npy")

# LightGBM - GPU - 5th Diverse Model (3-Seed Averaged) - 0.91582
LightGBM uses a different tree-building algorithm (leaf-wise vs XGBoost's level-wise, and different from CatBoost's symmetric trees). This structural diversity is key for ensemble gain. We use the same enhanced feature engineering, original data augmentation, and 3-seed averaging.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

VER = 500

import numpy as np
import pandas as pd
import gc
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# -----------------------------
# CONFIG
# -----------------------------
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
ORIG_PATH  = "/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv"

TARGET = "Churn"
DROP_COLS = ["customerID", "id"]

N_SPLITS = 5
SEEDS = [42, 2024, 7777]

# -----------------------------
# LOAD + ORIGINAL DATA
# -----------------------------
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

try:
    orig_df = pd.read_csv(ORIG_PATH)
    if TARGET in orig_df.columns:
        common_cols = [c for c in train_df.columns if c in orig_df.columns]
        orig_df = orig_df[common_cols].copy()
        for c in train_df.columns:
            if c not in orig_df.columns:
                orig_df[c] = np.nan
        orig_df = orig_df[train_df.columns]
        train_df = pd.concat([train_df, orig_df], axis=0, ignore_index=True)
        print(f"Original data loaded! Combined train shape: {train_df.shape}")
except Exception as e:
    print(f"No original data ({e}), using competition data only.")

y = (train_df[TARGET].astype(str).str.strip().str.lower() == "yes").astype(np.int32).values

# -----------------------------
# PREPROCESSING (same enhanced FE)
# -----------------------------
def preprocess_lgbm(X_tr: pd.DataFrame, X_te: pd.DataFrame):
    X_tr = X_tr.copy()
    X_te = X_te.copy()
    X_tr = X_tr.drop(columns=[c for c in DROP_COLS if c in X_tr.columns], errors="ignore")
    X_te = X_te.drop(columns=[c for c in DROP_COLS if c in X_te.columns], errors="ignore")

    n_tr = len(X_tr)
    X_all = pd.concat([X_tr, X_te], axis=0, ignore_index=True)

    # Numeric fixes
    if "TotalCharges" in X_all.columns:
        X_all["TotalCharges"] = pd.to_numeric(X_all["TotalCharges"], errors="coerce")
        mask = X_all["TotalCharges"].isna()
        X_all.loc[mask, "TotalCharges"] = (
            X_all.loc[mask, "tenure"].fillna(0) * X_all.loc[mask, "MonthlyCharges"].fillna(0)
        )
        X_all["TotalCharges"] = X_all["TotalCharges"].fillna(0.0).astype("float32")
    for c in ["tenure", "MonthlyCharges"]:
        if c in X_all.columns:
            X_all[c] = pd.to_numeric(X_all[c], errors="coerce")

    # Enhanced Feature Engineering (identical to XGBoost/CatBoost)
    X_all["tenure_x_monthly"] = X_all["tenure"] * X_all["MonthlyCharges"]
    X_all["avg_monthly"] = X_all["TotalCharges"] / (X_all["tenure"] + 1)
    X_all["charge_diff"] = X_all["MonthlyCharges"] - X_all["avg_monthly"]
    X_all["charge_ratio"] = X_all["MonthlyCharges"] / (X_all["TotalCharges"] + 1)

    X_all["tenure_sq"] = X_all["tenure"] ** 2
    X_all["monthly_sq"] = X_all["MonthlyCharges"] ** 2
    X_all["log_tenure"] = np.log1p(X_all["tenure"])
    X_all["log_total"] = np.log1p(X_all["TotalCharges"])
    X_all["log_monthly"] = np.log1p(X_all["MonthlyCharges"])

    service_cols = ["PhoneService","MultipleLines","OnlineSecurity","OnlineBackup",
                    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
    svc_present = [c for c in service_cols if c in X_all.columns]
    for c in svc_present:
        X_all[f"{c}_flag"] = (X_all[c].astype(str).str.lower() == "yes").astype(np.int8)
    X_all["n_services"] = X_all[[f"{c}_flag" for c in svc_present]].sum(axis=1)

    prot_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
    pc_present = [c for c in prot_cols if c in X_all.columns]
    X_all["n_protection"] = X_all[[f"{c}_flag" for c in pc_present]].sum(axis=1)

    stream_cols = ["StreamingTV", "StreamingMovies"]
    sc_present = [c for c in stream_cols if c in X_all.columns]
    X_all["n_streaming"] = X_all[[f"{c}_flag" for c in sc_present]].sum(axis=1)
    X_all = X_all.drop(columns=[f"{c}_flag" for c in svc_present])

    X_all["monthly_per_service"] = X_all["MonthlyCharges"] / (X_all["n_services"] + 1)

    if "InternetService" in X_all.columns:
        X_all["has_internet"] = (X_all["InternetService"].astype(str).str.lower() != "no").astype(np.int8)
        X_all["has_fiber"] = (X_all["InternetService"].astype(str).str.lower() == "fiber optic").astype(np.int8)

    if "Contract" in X_all.columns:
        contract_map = {"month-to-month": 0, "one year": 1, "two year": 2}
        X_all["contract_num"] = X_all["Contract"].astype(str).str.lower().map(contract_map).fillna(0).astype(np.int8)
        X_all["contract_x_tenure"] = X_all["contract_num"] * X_all["tenure"]
        X_all["contract_x_monthly"] = X_all["contract_num"] * X_all["MonthlyCharges"]

    if "PaymentMethod" in X_all.columns:
        freq = X_all["PaymentMethod"].value_counts(normalize=True)
        X_all["payment_freq"] = X_all["PaymentMethod"].map(freq).astype(np.float32)

    X_all["tenure_bin"] = pd.cut(X_all["tenure"], bins=[-1,6,12,24,48,72,999], labels=False).astype("float32")

    # LightGBM native categorical: convert object cols to 'category' dtype
    cat_cols = list(X_all.select_dtypes(include=["object"]).columns)
    for c in cat_cols:
        X_all[c] = X_all[c].astype(str).fillna("__MISSING__").astype("category")

    X_tr2 = X_all.iloc[:n_tr].reset_index(drop=True)
    X_te2 = X_all.iloc[n_tr:].reset_index(drop=True)
    return X_tr2, X_te2, cat_cols

X_train_raw = train_df.drop(columns=[TARGET]).copy()
X_test_raw  = test_df.copy()
X_train, X_test, cat_cols = preprocess_lgbm(X_train_raw, X_test_raw)
print(f"LightGBM features: {X_train.shape[1]}, categorical: {len(cat_cols)}")

# -----------------------------
# LGBM PARAMS
# -----------------------------
lgb_params = dict(
    objective="binary",
    metric="auc",
    n_estimators=100_000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.65,
    reg_alpha=0.1,
    reg_lambda=1.0,
    device="gpu",
    gpu_use_dp=False,
    verbose=-1,
)

# -----------------------------
# 3-SEED AVERAGED CV
# -----------------------------
oof_accum = np.zeros(len(X_train), dtype=np.float64)
test_accum = np.zeros(len(X_test), dtype=np.float64)

for seed_idx, SEED in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"SEED {seed_idx+1}/{len(SEEDS)}: {SEED}")
    print(f"{'='*60}")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_seed = np.zeros(len(X_train), dtype=np.float32)
    test_seed = np.zeros(len(X_test), dtype=np.float32)
    fold_aucs = []

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y), 1):
        X_trn, X_val = X_train.iloc[trn_idx], X_train.iloc[val_idx]
        y_trn, y_val = y[trn_idx], y[val_idx]

        model = lgb.LGBMClassifier(**lgb_params, random_state=SEED)
        model.fit(
            X_trn, y_trn,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=150, verbose=False),
                lgb.log_evaluation(period=500),
            ],
        )

        p_val = model.predict_proba(X_val)[:, 1].astype(np.float32)
        oof_seed[val_idx] = p_val

        auc = roc_auc_score(y_val, p_val)
        fold_aucs.append(auc)
        print(f"  Fold {fold}/{N_SPLITS} AUC = {auc:.6f}")

        test_seed += model.predict_proba(X_test)[:, 1].astype(np.float32) / N_SPLITS
        del model; gc.collect()

    seed_auc = roc_auc_score(y, oof_seed)
    print(f"  Seed {SEED} OOF AUC: {seed_auc:.6f}")

    oof_accum += oof_seed
    test_accum += test_seed

oof = (oof_accum / len(SEEDS)).astype(np.float32)
test_pred = (test_accum / len(SEEDS)).astype(np.float32)

final_auc = roc_auc_score(y, oof)
print(f"\n{'='*60}")
print(f"3-Seed Averaged LightGBM OOF AUC: {final_auc:.6f}")
print(f"{'='*60}")

np.save(f"oof_lgbm_v{VER}.npy", oof)
np.save(f"test_lgbm_v{VER}.npy", test_pred)
print(f"\nSaved: oof_lgbm_v{VER}.npy, test_lgbm_v{VER}.npy")

# Ensemble - GPU - CV 0.9178 -> 0.9182
The individual models achieves CVs 0.9160, 0.9166, and 0.9164. Since they are diverse, they combine well and their blend achieves CV 0.9178, wow!

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression, Ridge

# Reload target
train_ens = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
y = (train_ens["Churn"].astype(str).str.strip().str.lower() == "yes").astype(np.int32).values

# Load all 5 OOF predictions
logreg_oof = np.load("oof_tepair_logit3_v200.npy")
xgb_oof = np.load("oof_pred_proba_v103.npy")
catboost_oof = np.load("oof_catboost_v401.npy")
lgbm_oof = np.load("oof_lgbm_v500.npy")
mlp_oof = np.load("oof_nn_v312.npy")

# Ensure all predictions are same length, since some models are using data augmented with original dataset
N = len(y)
logreg_oof = logreg_oof[:N]
xgb_oof = xgb_oof[:N]
catboost_oof = catboost_oof[:N]
lgbm_oof = lgbm_oof[:N]
mlp_oof = mlp_oof[:N]

model_names = ["logreg", "xgb", "catboost", "lgbm", "mlp"]
all_oofs = [logreg_oof, xgb_oof, catboost_oof, lgbm_oof, mlp_oof]
N_MODELS = len(all_oofs)

# Print individual model AUCs
print("="*60)
print("Individual Model OOF AUCs:")
for name, oof in zip(model_names, all_oofs):
    print(f"  {name:>10s}: {roc_auc_score(y, oof):.6f}")
print("="*60)

# =================================================================
# STRATEGY 1: Simple average
# =================================================================
simple_avg = np.mean(all_oofs, axis=0)
simple_auc = roc_auc_score(y, simple_avg)
print(f"\n1. Simple Average       CV AUC = {simple_auc:.6f}")

# =================================================================
# STRATEGY 2: Rank average
# =================================================================
def rank_avg(*arrays):
    ranks = [rankdata(a) / len(a) for a in arrays]
    return np.mean(ranks, axis=0)

rank_ensemble = rank_avg(*all_oofs)
rank_auc = roc_auc_score(y, rank_ensemble)
print(f"2. Rank Average         CV AUC = {rank_auc:.6f}")

# =================================================================
# STRATEGY 3: Optimized weights (raw)
# =================================================================
def neg_auc(weights, oofs, y_true):
    w = np.abs(np.array(weights))
    w = w / w.sum()
    blend = sum(w[i] * oofs[i] for i in range(len(oofs)))
    return -roc_auc_score(y_true, blend)

result = minimize(neg_auc, x0=[1]*N_MODELS, args=(all_oofs, y),
                  method="Nelder-Mead", options={"maxiter": 100000, "xatol": 1e-8, "fatol": 1e-8})
best_w_raw = np.abs(np.array(result.x))
best_w_raw /= best_w_raw.sum()
opt_raw_auc = -result.fun
print(f"3. Optimized Weights    CV AUC = {opt_raw_auc:.6f}")
for n, w in zip(model_names, best_w_raw):
    print(f"     {n}: {w:.4f}")

# =================================================================
# STRATEGY 4: Optimized weights (rank)
# =================================================================
all_ranks = [rankdata(a) / len(a) for a in all_oofs]
result_r = minimize(neg_auc, x0=[1]*N_MODELS, args=(all_ranks, y),
                    method="Nelder-Mead", options={"maxiter": 100000, "xatol": 1e-8, "fatol": 1e-8})
best_w_rank = np.abs(np.array(result_r.x))
best_w_rank /= best_w_rank.sum()
opt_rank_auc = -result_r.fun
print(f"4. Opt Weights (rank)   CV AUC = {opt_rank_auc:.6f}")

# =================================================================
# STRATEGY 5: Stacking meta-learner (Logistic Regression)
# =================================================================
X_stack = np.column_stack(all_oofs)
X_stack_rank = np.column_stack(all_ranks)

# Add pairwise products as interaction features
interactions = []
interact_names = []
for i in range(N_MODELS):
    for j in range(i+1, N_MODELS):
        interactions.append(all_oofs[i] * all_oofs[j])
        interact_names.append(f"{model_names[i]}_x_{model_names[j]}")
X_stack_interact = np.column_stack([X_stack] + interactions)

skf_meta = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 5a: LogisticRegression stacking
oof_stack_lr = np.zeros(len(y), dtype=np.float64)
for tr_idx, va_idx in skf_meta.split(X_stack_interact, y):
    meta_lr = LogisticRegression(C=1.0, max_iter=5000, solver="lbfgs")
    meta_lr.fit(X_stack_interact[tr_idx], y[tr_idx])
    oof_stack_lr[va_idx] = meta_lr.predict_proba(X_stack_interact[va_idx])[:, 1]
stack_lr_auc = roc_auc_score(y, oof_stack_lr)
print(f"5a. Stack LR (interact) CV AUC = {stack_lr_auc:.6f}")

# 5b: Ridge stacking on rank features
interactions_rank = []
for i in range(N_MODELS):
    for j in range(i+1, N_MODELS):
        interactions_rank.append(all_ranks[i] * all_ranks[j])
X_stack_rank_interact = np.column_stack([X_stack_rank] + interactions_rank)

oof_stack_ridge = np.zeros(len(y), dtype=np.float64)
for tr_idx, va_idx in skf_meta.split(X_stack_rank_interact, y):
    meta_ridge = Ridge(alpha=1.0)
    meta_ridge.fit(X_stack_rank_interact[tr_idx], y[tr_idx])
    oof_stack_ridge[va_idx] = meta_ridge.predict(X_stack_rank_interact[va_idx])
stack_ridge_auc = roc_auc_score(y, oof_stack_ridge)
print(f"5b. Stack Ridge (rank)  CV AUC = {stack_ridge_auc:.6f}")

# 5c: LR stacking on raw (no interactions)
oof_stack_lr_simple = np.zeros(len(y), dtype=np.float64)
for tr_idx, va_idx in skf_meta.split(X_stack, y):
    meta_lr2 = LogisticRegression(C=1.0, max_iter=5000, solver="lbfgs")
    meta_lr2.fit(X_stack[tr_idx], y[tr_idx])
    oof_stack_lr_simple[va_idx] = meta_lr2.predict_proba(X_stack[va_idx])[:, 1]
stack_lr_simple_auc = roc_auc_score(y, oof_stack_lr_simple)
print(f"5c. Stack LR (simple)   CV AUC = {stack_lr_simple_auc:.6f}")

# =================================================================
# PICK THE BEST STRATEGY
# =================================================================
strategies = {
    "simple_avg": simple_auc,
    "rank_avg": rank_auc,
    "opt_raw": opt_raw_auc,
    "opt_rank": opt_rank_auc,
    "stack_lr_interact": stack_lr_auc,
    "stack_ridge_rank": stack_ridge_auc,
    "stack_lr_simple": stack_lr_simple_auc,
}

print(f"\n{'='*60}")
print("STRATEGY COMPARISON:")
for name, auc in sorted(strategies.items(), key=lambda x: -x[1]):
    marker = " <-- BEST" if auc == max(strategies.values()) else ""
    print(f"  {name:>22s}: {auc:.6f}{marker}")

best_strategy = max(strategies, key=strategies.get)
print(f"\nBest strategy: {best_strategy} (AUC = {strategies[best_strategy]:.6f})")
print(f"{'='*60}")

# Create Submission CSV

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression, Ridge

sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv")

# Load all 5 test predictions
logreg_pred = np.load("pred_tepair_logit3_v200.npy")
xgb_pred = np.load("test_pred_proba_v103.npy")
catboost_pred = np.load("test_catboost_v401.npy")
lgbm_pred = np.load("test_lgbm_v500.npy")
mlp_pred = np.load("pred_nn_v312.npy")

all_test_preds = [logreg_pred, xgb_pred, catboost_pred, lgbm_pred, mlp_pred]
N_MODELS = len(all_test_preds)

def rank_avg(*arrays):
    ranks = [rankdata(a) / len(a) for a in arrays]
    return np.mean(ranks, axis=0)

# Apply the best strategy from ensemble evaluation
if best_strategy == "stack_lr_interact":
    # Retrain on full OOF data and predict test
    X_stack_full = np.column_stack(all_oofs)
    interactions_full = []
    for i in range(N_MODELS):
        for j in range(i+1, N_MODELS):
            interactions_full.append(all_oofs[i] * all_oofs[j])
    X_stack_full = np.column_stack([X_stack_full] + interactions_full)

    X_test_stack = np.column_stack(all_test_preds)
    interactions_test = []
    for i in range(N_MODELS):
        for j in range(i+1, N_MODELS):
            interactions_test.append(all_test_preds[i] * all_test_preds[j])
    X_test_stack = np.column_stack([X_test_stack] + interactions_test)

    meta_final = LogisticRegression(C=1.0, max_iter=5000, solver="lbfgs")
    meta_final.fit(X_stack_full, y)
    pred = meta_final.predict_proba(X_test_stack)[:, 1]

elif best_strategy == "stack_ridge_rank":
    all_oof_ranks = [rankdata(a) / len(a) for a in all_oofs]
    X_stack_full = np.column_stack(all_oof_ranks)
    interactions_full = []
    for i in range(N_MODELS):
        for j in range(i+1, N_MODELS):
            interactions_full.append(all_oof_ranks[i] * all_oof_ranks[j])
    X_stack_full = np.column_stack([X_stack_full] + interactions_full)

    all_test_ranks = [rankdata(a) / len(a) for a in all_test_preds]
    X_test_stack = np.column_stack(all_test_ranks)
    interactions_test = []
    for i in range(N_MODELS):
        for j in range(i+1, N_MODELS):
            interactions_test.append(all_test_ranks[i] * all_test_ranks[j])
    X_test_stack = np.column_stack([X_test_stack] + interactions_test)

    meta_final = Ridge(alpha=1.0)
    meta_final.fit(X_stack_full, y)
    pred = meta_final.predict(X_test_stack)

elif best_strategy == "stack_lr_simple":
    X_stack_full = np.column_stack(all_oofs)
    X_test_stack = np.column_stack(all_test_preds)
    meta_final = LogisticRegression(C=1.0, max_iter=5000, solver="lbfgs")
    meta_final.fit(X_stack_full, y)
    pred = meta_final.predict_proba(X_test_stack)[:, 1]

elif best_strategy == "opt_rank":
    test_ranks = [rankdata(a) / len(a) for a in all_test_preds]
    pred = sum(best_w_rank[i] * test_ranks[i] for i in range(N_MODELS))

elif best_strategy == "opt_raw":
    pred = sum(best_w_raw[i] * all_test_preds[i] for i in range(N_MODELS))

elif best_strategy == "rank_avg":
    pred = rank_avg(*all_test_preds)

else:  # simple_avg
    pred = np.mean(all_test_preds, axis=0)

print(f"Submission using strategy: {best_strategy}")
print(f"pred min={pred.min():.4f}, max={pred.max():.4f}, mean={pred.mean():.4f}")

sub['Churn'] = pred
sub.to_csv("submission.csv", index=False)
sub.head()